# MicroFinML - Model Evaluation & Comparison
## Phase 4: Evaluate All Models on Test Set

**Evaluation Metrics:**
- Accuracy, Precision, Recall, F1-Score, ROC-AUC
- Confusion Matrices
- ROC Curves & Precision-Recall Curves
- Feature Importance Analysis
- Final Model Recommendation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys, os
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

from src.model_evaluation import (
    evaluate_all_models, generate_classification_reports,
    plot_confusion_matrices, plot_roc_curves,
    plot_precision_recall_curves, plot_feature_importance,
    plot_model_comparison, save_metrics, full_evaluation
)

print('Modules loaded successfully!')

## 1. Load Models and Test Data

In [ ]:
# Load processed test data
processed_dir = '../data/processed'
X_test = pd.read_csv(os.path.join(processed_dir, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(processed_dir, 'y_test.csv')).squeeze()
feature_names = X_test.columns.tolist()

# Load trained models
models_dict = {
    'LogisticRegression': joblib.load('../models/logisticregression_model.pkl'),
    'RandomForest': joblib.load('../models/randomforest_model.pkl'),
    'XGBoost': joblib.load('../models/xgboost_model.pkl')
}

print(f'Test set: {X_test.shape[0]:,} samples, {X_test.shape[1]} features')
print(f'Test default rate: {y_test.mean():.4f}')
print(f'Models loaded: {list(models_dict.keys())}')

## 2. Evaluate All Models

In [ ]:
metrics_df, predictions = evaluate_all_models(models_dict, X_test, y_test)
print('\n=== Performance Metrics (Test Set) ===')
metrics_df.round(4)

## 3. Classification Reports

In [ ]:
reports = generate_classification_reports(models_dict, X_test, y_test)

## 4. Confusion Matrices

In [ ]:
plot_confusion_matrices(models_dict, X_test, y_test,
                        save_dir='../results/figures/model_comparison')

## 5. ROC Curves

In [ ]:
plot_roc_curves(models_dict, X_test, y_test,
                save_dir='../results/figures/model_comparison')

## 6. Precision-Recall Curves

In [ ]:
plot_precision_recall_curves(models_dict, X_test, y_test,
                              save_dir='../results/figures/model_comparison')

## 7. Model Comparison

In [ ]:
plot_model_comparison(metrics_df, save_dir='../results/figures/model_comparison')

# Save metrics to CSV
save_metrics(metrics_df, '../results/metrics/model_performance.csv')

## 8. Feature Importance

In [ ]:
feat_dir = '../results/figures/feature_importance'

for name, model in models_dict.items():
    print(f'\n--- {name} ---')
    feat_imp = plot_feature_importance(model, feature_names, name, top_n=20, save_dir=feat_dir)
    if feat_imp is not None:
        print(feat_imp.head(10).to_string(index=False))

## 9. Best Model Selection

In [ ]:
best_model_name = metrics_df['roc_auc'].idxmax()
best_metrics = metrics_df.loc[best_model_name]

print('=' * 60)
print('BEST MODEL RECOMMENDATION')
print('=' * 60)
print(f'\nBest Model: {best_model_name}')
print(f'\nPerformance on Test Set:')
print(f'  Accuracy:  {best_metrics["accuracy"]:.4f}')
print(f'  Precision: {best_metrics["precision"]:.4f}')
print(f'  Recall:    {best_metrics["recall"]:.4f}')
print(f'  F1-Score:  {best_metrics["f1_score"]:.4f}')
print(f'  ROC-AUC:   {best_metrics["roc_auc"]:.4f}')
print(f'\nThis model is recommended for deployment in the')
print(f'MicroFinML loan default prediction system.')
print('=' * 60)

## 10. Demo: Predict on New Applicant

In [ ]:
from src.prediction import predict_single, load_model, load_preprocessor

# Load best model and preprocessor
best_model = models_dict[best_model_name]
preprocessor = joblib.load('../data/processed/preprocessor.pkl')

# Sample applicant
sample_applicant = {
    'Age': 35,
    'Income': 55000,
    'LoanAmount': 25000,
    'CreditScore': 680,
    'MonthsEmployed': 48,
    'NumCreditLines': 3,
    'InterestRate': 12.5,
    'LoanTerm': 36,
    'DTIRatio': 0.35,
    'Education': "Bachelor's",
    'EmploymentType': 'Full-time',
    'MaritalStatus': 'Married',
    'HasMortgage': 'Yes',
    'HasDependents': 'Yes',
    'LoanPurpose': 'Home',
    'HasCoSigner': 'No'
}

result = predict_single(sample_applicant, best_model, preprocessor)

print('\n' + '=' * 50)
print('LOAN DEFAULT PREDICTION - SAMPLE APPLICANT')
print('=' * 50)
print(f"Prediction:          {result['prediction_label']}")
print(f"Default Probability: {result['default_probability']:.2%}")
print(f"Repay Probability:   {result['repay_probability']:.2%}")
print(f"Risk Level:          {result['risk_level']}")
print('=' * 50)

## Conclusion

### Summary
We successfully built and evaluated three machine learning models for loan default prediction:

1. **Logistic Regression** - Simple baseline model
2. **Random Forest** - Ensemble of decision trees
3. **XGBoost** - Gradient boosting (typically best performer)

### Key Findings
- The models were trained on 255K+ loan records with engineered features
- Class imbalance (88% vs 12%) was addressed using SMOTE
- All models were evaluated using multiple metrics for robust comparison
- Feature importance analysis reveals which factors most influence default risk

### Impact
This ML system can help microfinance institutions:
- **Reduce default risk** by identifying high-risk applicants
- **Make faster decisions** with automated credit scoring
- **Support sustainable lending** by optimizing loan approvals
- **Promote financial inclusion** through data-driven decision making